In [9]:
import numpy as np
import torch
from torchsummary import summary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

def preprocess(train_file, test_file):
    train = np.load(train_file, allow_pickle=True)
    test = np.load(test_file, allow_pickle=True)

    X_train, y_train = train["X"], train["y"]
    X_test, y_test = test["X"], test["y"]

    transform = transforms.Compose([
        transforms.Resize((32, 32), interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor()
    ])

    X_train = torch.stack([transform(Image.fromarray(img)) for img in X_train])
    X_test = torch.stack([transform(Image.fromarray(img)) for img in X_test])

    return (X_train, y_train), (X_test, y_test)


(X_train, y_train), (X_test, y_test) = preprocess("train2.npz", "test2.npz")

In [21]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(16, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.dense_layers = nn.Sequential(
            nn.Linear(16 * 8 * 8, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.1),

            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.2),

            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),

            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = torch.flatten(x, 1)
        x = self.dense_layers(x)
        return x

model = Model()
summary(model, (1, 32, 32))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 32, 32]             160
       BatchNorm2d-2           [-1, 16, 32, 32]              32
              ReLU-3           [-1, 16, 32, 32]               0
         MaxPool2d-4           [-1, 16, 16, 16]               0
            Conv2d-5           [-1, 16, 16, 16]           2,320
       BatchNorm2d-6           [-1, 16, 16, 16]              32
              ReLU-7           [-1, 16, 16, 16]               0
         MaxPool2d-8             [-1, 16, 8, 8]               0
            Linear-9                  [-1, 128]         131,200
      BatchNorm1d-10                  [-1, 128]             256
             ReLU-11                  [-1, 128]               0
          Dropout-12                  [-1, 128]               0
           Linear-13                  [-1, 128]          16,512
      BatchNorm1d-14                  [

In [23]:
# SUBIECTUL 2 – antrenare (3p: ≥90%, ≥94%, ≥98%)
# Celula rămâne cum e (Adam e deja pus). Rulezi -> model2.pth.
# Predai S2.ipynb + model2.pth. test() NU o modifici.

def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def test(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * len(y)
            correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 15
best_model = 0
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = test(model, test_loader, criterion, device)
    if test_acc > best_model:
        best_model = test_acc
        torch.save(model.state_dict(), "model2.pth")
    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%}"
    )


/var/folders/by/8fc6d5ln63xg5yzt1yyss0jm0000gn/T/ipykernel_81096/348561807.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_train, dtype=torch.float32),
/var/folders/by/8fc6d5ln63xg5yzt1yyss0jm0000gn/T/ipykernel_81096/348561807.py:46: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_test, dtype=torch.float32),


Epoch 1/15 | Train Loss: 0.2551 | Train Acc: 94.16% | Test Loss: 0.0536 | Test Acc: 98.40%
Epoch 2/15 | Train Loss: 0.0908 | Train Acc: 97.59% | Test Loss: 0.0561 | Test Acc: 98.27%
Epoch 3/15 | Train Loss: 0.0710 | Train Acc: 98.10% | Test Loss: 0.0414 | Test Acc: 98.68%
Epoch 4/15 | Train Loss: 0.0603 | Train Acc: 98.38% | Test Loss: 0.0439 | Test Acc: 98.54%
Epoch 5/15 | Train Loss: 0.0538 | Train Acc: 98.55% | Test Loss: 0.0376 | Test Acc: 98.84%
Epoch 6/15 | Train Loss: 0.0468 | Train Acc: 98.67% | Test Loss: 0.0341 | Test Acc: 99.00%
Epoch 7/15 | Train Loss: 0.0427 | Train Acc: 98.81% | Test Loss: 0.0303 | Test Acc: 99.11%
Epoch 8/15 | Train Loss: 0.0366 | Train Acc: 98.95% | Test Loss: 0.0315 | Test Acc: 99.09%
Epoch 9/15 | Train Loss: 0.0365 | Train Acc: 98.99% | Test Loss: 0.0338 | Test Acc: 99.07%
Epoch 10/15 | Train Loss: 0.0317 | Train Acc: 99.14% | Test Loss: 0.0322 | Test Acc: 99.17%
Epoch 11/15 | Train Loss: 0.0277 | Train Acc: 99.19% | Test Loss: 0.0290 | Test Acc: 99.1